# trening i testowanie modeli.

##### Collaborative filtering with matrix factorization

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
import mlflow
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.models import MF_SGD, MF_SVD, cf_to_numpy, prepare_cf_data

In [2]:
df_ratings = pd.read_csv("../data/processed/ratings.csv")
df_movies = pd.read_csv("../data/processed/movies.csv")
print("Number of ratings before filtering:", len(df_ratings))

# leave only movies and users with at least 20 ratings
min_movie_ratings = 20
min_user_ratings = 20

movie_counts = df_ratings.groupby('movieId').size()
df_ratings = df_ratings[df_ratings['movieId'].isin(movie_counts[movie_counts >= min_movie_ratings].index)]
user_counts = df_ratings.groupby('userId').size()
df_ratings = df_ratings[df_ratings['userId'].isin(user_counts[user_counts >= min_user_ratings].index)]

print("Number of ratings after filtering:", len(df_ratings))

train_df, test_df, user_map, movie_map = prepare_cf_data(df_ratings)

(train_users, train_items, train_ratings, test_users, test_items, test_ratings) = cf_to_numpy(train_df, test_df)

Number of ratings before filtering: 31957234
Number of ratings after filtering: 31678415


In [ ]:
k = 20
epochs = 3
lr = 0.01
reg = 0.02

model = MF_SGD(k=k, epochs=epochs, lr=lr, reg=reg)

model.fit(
    train_users,
    train_items,
    train_ratings,
    len(user_map),
    len(movie_map)
)

[MF-SGD] Epoch 1: RMSE = 0.8884
[MF-SGD] Epoch 2: RMSE = 0.8531
[MF-SGD] Epoch 3: RMSE = 0.8271


In [26]:
# train_df, test_df, user_map, movie_map = prepare_cf_data(df_ratings)

# (train_users, train_items, train_ratings, test_users, test_items, test_ratings) = cf_to_numpy(train_df, test_df)
# k = 20
# epochs = 3
# lr = 0.01
# reg = 0.02

# mlflow.set_experiment("mf-sgd")

# with mlflow.start_run(run_name=f"MF-SGD_k{k}"):

#     mlflow.log_param("k", k)
#     mlflow.log_param("epochs", epochs)
#     mlflow.log_param("lr", lr)
#     mlflow.log_param("reg", reg)

#     model = MF_SGD(k=k, epochs=epochs, lr=lr, reg=reg)

#     model.fit(
#         train_users,
#         train_items,
#         train_ratings,
#         len(user_map),
#         len(movie_map)
#     )

#     preds = model.predict_batch(test_users, test_items)

#     rmse = np.sqrt(np.mean((test_ratings - preds) ** 2))

#     mlflow.log_metric("rmse", rmse)

#     print(f"RMSE: {rmse:.4f}")

In [44]:
user_id = list(user_map.keys())[1]

recommendations = model.recommend(
    user_id=user_id,
    user_map=user_map,
    movie_map=movie_map,
    df_movies=df_movies,
    top_n=5
)

print(f"Top 5 recommendations for user {user_id}:")
recommendations

Top 5 recommendations for user 79017:


,title
834,"Godfather, The (1972)"
877,Rear Window (1954)
1913,Seven Samurai (Shichinin no samurai) (1954)
6517,Ikiru (1952)
46131,Planet Earth II (2016)


In [ ]:
# SVD Experiment
k_factors = 50
user_id = list(user_map.keys())[1]
mlflow.set_experiment("Movie_Recommender_SVD")

with mlflow.start_run(run_name=f"SVD_k{k_factors}"):
    svd_model = MF_SVD(k=k_factors)
    print(f"Starting SVD training...")
    svd_model.fit(
        train_df=train_df, 
        n_users=len(user_map), 
        n_movies=len(movie_map)
    )
    
    print("Evaluating SVD...")
    svd_preds = svd_model.predict_batch(test_users, test_items)
    svd_rmse = np.sqrt(np.mean((test_ratings - svd_preds)**2))
    
    mlflow.log_param("k", k_factors)
    mlflow.log_metric("rmse", svd_rmse)
    mlflow.sklearn.log_model(svd_model, "svd_model")
    
    print(f"SVD Training finished. RMSE: {svd_rmse:.4f}")

# 5. Recommendations for the SAME user as SGD
recommendations_svd = svd_model.recommend(
    user_id=user_id, 
    user_map=user_map,
    movie_map=movie_map,
    df_movies=df_movies,
    top_n=5
)

print(f"\nSVD Top 5 for user {user_id}:")
recommendations_svd

Starting SVD training...
Evaluating SVD...


2026/05/02 18:43:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 18:43:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/02 18:43:13 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


SVD Training finished. RMSE: 0.8845

SVD Top 5 for user 79017:


,title
520,Schindler's List (1993)
1157,Star Wars: Episode V - The Empire Strikes Back...
4860,"Lord of the Rings: The Fellowship of the Ring,..."
5811,"Lord of the Rings: The Two Towers, The (2002)"
6998,"Lord of the Rings: The Return of the King, The..."
